In [70]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [71]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df.dropna(inplace=True)
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-06-27,95.385231,96.672957,94.946656,96.672957
1,2016-06-28,97.475441,97.522101,96.393006,96.439659
2,2016-06-29,99.201744,99.407035,98.193962,98.240615
3,2016-06-30,100.349495,100.414814,99.033769,99.369699
4,2016-07-01,100.853416,101.226673,100.265536,100.302862
...,...,...,...,...,...
2508,2026-06-18,739.807007,741.005702,731.705924,736.390778
2509,2026-06-22,737.950012,745.450012,734.390015,742.020020
2510,2026-06-23,713.650024,723.609985,712.109985,715.739990
2511,2026-06-24,710.619995,719.929993,704.450012,715.369995


In [72]:
df['Momentum'] = (df['Close'] / df['Close'].shift(5) - 1) * 100
df.dropna(inplace=True)
df

Price,Date,Close,High,Low,Open,Momentum
5,2016-07-05,100.237511,100.461469,99.696290,100.321497,5.087035
6,2016-07-06,101.058701,101.105353,99.444371,99.761641,3.676064
7,2016-07-07,101.357285,101.581235,100.881383,101.151993,2.172886
8,2016-07-08,102.924957,102.980943,101.730537,101.898501,2.566493
9,2016-07-11,103.512848,103.895430,103.260904,103.270230,2.636927
...,...,...,...,...,...,...
2508,2026-06-18,739.807007,741.005702,731.705924,736.390778,3.277003
2509,2026-06-22,737.950012,745.450012,734.390015,742.020020,2.415086
2510,2026-06-23,713.650024,723.609985,712.109985,715.739990,-3.973887
2511,2026-06-24,710.619995,719.929993,704.450012,715.369995,-2.529121


In [73]:
def signal(data):
    signal = [0] * len(data)
    for i in range(2,len(data)):
        if (data['Momentum'].iloc[i] < -5):
            signal[i] = 1
        elif (data['Momentum'].iloc[i-1] < 0) and (data['Momentum'].iloc[i] > 0):
            signal[i] = 2
        else:
            signal[i] = 0
        data["signal"] = signal
        
signal(df)
df

Price,Date,Close,High,Low,Open,Momentum,signal
5,2016-07-05,100.237511,100.461469,99.696290,100.321497,5.087035,0
6,2016-07-06,101.058701,101.105353,99.444371,99.761641,3.676064,0
7,2016-07-07,101.357285,101.581235,100.881383,101.151993,2.172886,0
8,2016-07-08,102.924957,102.980943,101.730537,101.898501,2.566493,0
9,2016-07-11,103.512848,103.895430,103.260904,103.270230,2.636927,0
...,...,...,...,...,...,...,...
2508,2026-06-18,739.807007,741.005702,731.705924,736.390778,3.277003,0
2509,2026-06-22,737.950012,745.450012,734.390015,742.020020,2.415086,0
2510,2026-06-23,713.650024,723.609985,712.109985,715.739990,-3.973887,0
2511,2026-06-24,710.619995,719.929993,704.450012,715.369995,-2.529121,0


In [74]:
def oversold_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['oversold_entries'] = df.apply(lambda x: oversold_entries(x), axis=1)

def momentum_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['momentum_entries'] = df.apply(lambda x: momentum_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2187
2     230
1      91
Name: count, dtype: int64


(2508, 9)

In [75]:
df.set_index('Date', inplace=True)

In [76]:

bar = 2010
df1 = df[bar:bar+500].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['oversold_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Oversold Entries")

fig.add_scatter(x=df1.index, y=df1['momentum_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="gold"),
                name="Oversold Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.Momentum, 
                         line=dict(color='#00df9a', width=2),
                         name='Momentum'),
                         row=2, col=1)

fig.add_hline(y=10, line_width=0.5, line_color="grey", row=2, col=1)
fig.add_hline(y=5, line_width=0.5, line_color="grey", row=2, col=1)
fig.add_hline(y=-5, line_width=0.5, line_color="grey", row=2, col=1)
fig.add_hline(y=-10, line_width=0.5, line_color="grey", row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [84]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.02*price)

        elif self.signal==2: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, sl=0.98*price, tp=1.035*price)

                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2507 [00:00<?, ?bar/s]

Start                     2016-07-05 00:00:00
End                       2026-06-25 00:00:00
Duration                   3642 days 00:00:00
Exposure Time [%]                    51.91388
Equity Final [$]                 498646.64668
Equity Peak [$]                  524788.66201
Commissions [$]                   42203.11169
Return [%]                          398.64665
Buy & Hold Return [%]               614.68256
Return (Ann.) [%]                    17.52037
Volatility (Ann.) [%]                19.32995
CAGR [%]                             11.75892
Sharpe Ratio                          0.90638
Sortino Ratio                         1.61636
Calmar Ratio                          0.73214
Alpha [%]                            68.28005
Beta                                  0.53746
Max. Drawdown [%]                   -23.93025
Avg. Drawdown [%]                    -2.22634
Max. Drawdown Duration      229 days 00:00:00
Avg. Drawdown Duration       19 days 00:00:00
# Trades                          

In [81]:
trades = stats['_trades']
trades

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,Entry_SIGNAL,Exit_SIGNAL,CumulativePnL
0,902,36,47,109.624882,107.505548,107.505548,113.539023,-2009.564673,97.925824,-0.020323,2016-08-24,2016-09-09,16 days,None,0,0,-2009.564673
1,884,52,85,109.656304,107.578709,107.578709,113.616290,-1932.611526,96.017876,-0.019937,2016-09-16,2016-11-02,47 days,None,0,0,-3942.176199
2,877,90,91,108.271819,107.362368,107.362368,113.387807,-892.144277,94.555591,-0.009396,2016-11-09,2016-11-10,1 days,None,0,0,-4834.320476
3,853,97,105,110.292478,107.958232,107.958232,114.017112,-2084.195974,93.083928,-0.022154,2016-11-18,2016-12-01,13 days,None,0,0,-6918.516450
4,831,110,129,110.769555,114.598076,108.508323,114.598076,3087.860029,93.640251,0.033546,2016-12-08,2017-01-06,29 days,None,0,0,-3830.656421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,833,2448,2448,563.670522,568.820519,NaN,568.820519,3818.264622,471.682519,0.008132,2026-03-31,2026-03-31,0 days,None,0,0,378479.100186
165,808,2451,2453,585.586476,608.041802,572.651073,604.789654,17661.677117,482.225824,0.037328,2026-04-06,2026-04-08,2 days,None,0,0,396140.777303
166,676,2486,2490,725.163128,741.838651,702.417273,741.838651,10776.806383,495.846601,0.021984,2026-05-26,2026-06-01,6 days,None,0,0,406917.583687
167,715,2497,2499,700.889750,721.194038,NaN,721.194038,14009.171191,508.394954,0.027955,2026-06-10,2026-06-12,2 days,None,1,2,420926.754878


In [80]:
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [82]:
def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='% / Trade',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()